In [ ]:
import pandas as pd
from pathlib import Path



In [2]:
# Project root
PROJECT_ROOT = Path.cwd().parent

# Cleaned dataset path
data_path = PROJECT_ROOT / "data" / "processed" / "purchases_clean.csv"

# Load cleaned data
purchases_clean = pd.read_csv(data_path)


In [3]:
purchases_clean


,InvoiceID,date,CustomerID,product_id,quantity
0,536365,2014-12-01,17850,3851,6
1,536365,2014-12-01,17850,3859,6
2,536365,2014-12-01,17850,885,8
3,536365,2014-12-01,17850,1873,6
4,536365,2014-12-01,17850,2871,6
...,...,...,...,...,...
431166,11361,2015-12-30,4058,1036,4
431167,5596,2015-12-30,2507,837,2
431168,1874,2015-12-30,1509,1989,2
431169,9540,2015-12-30,3562,1258,3


In [4]:

# Basic verification
print("Shape:", purchases_clean.shape)
print("\nColumns:")
print(purchases_clean.columns.tolist())

print("\nData types:")
print(purchases_clean.dtypes)

print("\nMissing values:")
print(purchases_clean.isnull().sum())

print("\nDate range:")
print(purchases_clean["date"].min(), "to", purchases_clean["date"].max())

print("\nQuantity summary:")
print(purchases_clean["quantity"].describe())

print("\nFirst 5 rows:")
display(purchases_clean.head())

Shape: (431171, 5)

Columns:
['InvoiceID', 'date', 'CustomerID', 'product_id', 'quantity']

Data types:
InvoiceID      int64
date          object
CustomerID     int64
product_id     int64
quantity       int64
dtype: object

Missing values:
InvoiceID     0
date          0
CustomerID    0
product_id    0
quantity      0
dtype: int64

Date range:
2014-01-01 to 2015-12-30

Quantity summary:
count    431171.000000
mean         11.842399
std          45.426099
min           1.000000
25%           2.000000
50%           4.000000
75%          12.000000
max       12540.000000
Name: quantity, dtype: float64

First 5 rows:


,InvoiceID,date,CustomerID,product_id,quantity
0,536365,2014-12-01,17850,3851,6
1,536365,2014-12-01,17850,3859,6
2,536365,2014-12-01,17850,885,8
3,536365,2014-12-01,17850,1873,6
4,536365,2014-12-01,17850,2871,6


In [5]:
# Convert date to datetime
purchases_clean["date"] = pd.to_datetime(purchases_clean["date"])

# Check unique products and dates
print("Unique products:", purchases_clean["product_id"].nunique())
print("Unique dates:", purchases_clean["date"].nunique())

# Check transactions per date-product combination
date_product_counts = (
    purchases_clean
    .groupby(["date", "product_id"])
    .size()
)

print("\nDate-product combinations:", len(date_product_counts))
print("\nCombinations with multiple transaction rows:")
print((date_product_counts > 1).sum())

print("\nTarget definition:")
print("Grain  : date + product_id")
print("Target : daily total quantity")

Unique products: 4032
Unique dates: 728

Date-product combinations: 247902

Combinations with multiple transaction rows:
93979

Target definition:
Grain  : date + product_id
Target : daily total quantity


In [7]:
# Aggregate transactions to daily product demand
daily_demand = (
    purchases_clean
    .groupby(["date", "product_id"], as_index=False)["quantity"]
    .sum()
    .rename(columns={"quantity": "demand"})
)

# Verify
print("Shape:", daily_demand.shape)

print("\nColumns:")
print(daily_demand.columns.tolist())

print("\nMissing values:")
print(daily_demand.isnull().sum())

print("\nFirst 5 rows:")
display(daily_demand.head())

Shape: (247902, 3)

Columns:
['date', 'product_id', 'demand']

Missing values:
date          0
product_id    0
demand        0
dtype: int64

First 5 rows:


,date,product_id,demand
0,2014-01-01,307,4
1,2014-01-01,521,1
2,2014-01-01,522,11
3,2014-01-01,547,1
4,2014-01-01,565,1


In [8]:
# Aggregate transactions to daily product demand
daily_demand = (
    purchases_clean
    .groupby(["date", "product_id"], as_index=False)["quantity"]
    .sum()
    .rename(columns={"quantity": "demand"})
)

# Verify
print("Shape:", daily_demand.shape)

print("\nColumns:")
print(daily_demand.columns.tolist())

print("\nMissing values:")
print(daily_demand.isnull().sum())

print("\nFirst 5 rows:")
display(daily_demand.head())

Shape: (247902, 3)

Columns:
['date', 'product_id', 'demand']

Missing values:
date          0
product_id    0
demand        0
dtype: int64

First 5 rows:


,date,product_id,demand
0,2014-01-01,307,4
1,2014-01-01,521,1
2,2014-01-01,522,11
3,2014-01-01,547,1
4,2014-01-01,565,1


In [9]:
# Create calendar features
daily_demand["year"] = daily_demand["date"].dt.year
daily_demand["month"] = daily_demand["date"].dt.month
daily_demand["day"] = daily_demand["date"].dt.day
daily_demand["day_of_week"] = daily_demand["date"].dt.dayofweek
daily_demand["week_of_year"] = daily_demand["date"].dt.isocalendar().week.astype(int)
daily_demand["quarter"] = daily_demand["date"].dt.quarter
daily_demand["day_of_year"] = daily_demand["date"].dt.dayofyear

# Weekend indicator
daily_demand["is_weekend"] = (
    daily_demand["day_of_week"] >= 5
).astype(int)

# Verify
print("Columns:")
print(daily_demand.columns.tolist())

print("\nFirst 5 rows:")
display(daily_demand.head())

Columns:
['date', 'product_id', 'demand', 'year', 'month', 'day', 'day_of_week', 'week_of_year', 'quarter', 'day_of_year', 'is_weekend']

First 5 rows:


,date,product_id,demand,year,month,day,day_of_week,week_of_year,quarter,day_of_year,is_weekend
0,2014-01-01,307,4,2014,1,1,2,1,1,1,0
1,2014-01-01,521,1,2014,1,1,2,1,1,1,0
2,2014-01-01,522,11,2014,1,1,2,1,1,1,0
3,2014-01-01,547,1,2014,1,1,2,1,1,1,0
4,2014-01-01,565,1,2014,1,1,2,1,1,1,0


In [10]:
# Check the number of observed days per product
product_days = (
    daily_demand
    .groupby("product_id")["date"]
    .nunique()
)

print("Products:", product_days.shape[0])

print("\nObserved days per product:")
print(product_days.describe())

print("\nProducts with only 1 observed day:", (product_days == 1).sum())
print("Products with <= 7 observed days:", (product_days <= 7).sum())
print("Products with > 365 observed days:", (product_days > 365).sum())

Products: 4032

Observed days per product:
count    4032.000000
mean       61.483631
std        71.111676
min         1.000000
25%        10.000000
50%        38.000000
75%        88.000000
max       697.000000
Name: date, dtype: float64

Products with only 1 observed day: 223
Products with <= 7 observed days: 853
Products with > 365 observed days: 21


In [11]:
# Check overall calendar coverage
all_dates = pd.date_range(
    daily_demand["date"].min(),
    daily_demand["date"].max(),
    freq="D"
)

observed_dates = daily_demand["date"].unique()

missing_dates = all_dates.difference(observed_dates)

print("Expected calendar days:", len(all_dates))
print("Observed calendar days:", len(observed_dates))
print("Missing calendar days:", len(missing_dates))

if len(missing_dates) > 0:
    print("\nMissing dates:")
    print(missing_dates)
else:
    print("\nNo calendar dates are missing.")

Expected calendar days: 729
Observed calendar days: 728
Missing calendar days: 1

Missing dates:
DatetimeIndex(['2014-12-31'], dtype='datetime64[ns]', freq='D')


In [12]:
# Verify the missing calendar date
missing_date = pd.Timestamp("2014-12-31")

print("Transactions on 2014-12-31:",
      (purchases_clean["date"] == missing_date).sum())

print("Demand on 2014-12-31:",
      purchases_clean.loc[
          purchases_clean["date"] == missing_date,
          "quantity"
      ].sum())

Transactions on 2014-12-31: 0
Demand on 2014-12-31: 0


In [13]:
# Create a complete calendar for each product
all_dates = pd.date_range(
    daily_demand["date"].min(),
    daily_demand["date"].max(),
    freq="D"
)

all_products = daily_demand["product_id"].unique()

complete_index = pd.MultiIndex.from_product(
    [all_dates, all_products],
    names=["date", "product_id"]
)

# Reindex demand to the complete calendar
demand_calendar = (
    daily_demand[["date", "product_id", "demand"]]
    .set_index(["date", "product_id"])
    .reindex(complete_index)
    .reset_index()
)

print("Complete calendar shape:", demand_calendar.shape)

print("\nOriginal observed rows:", len(daily_demand))

print("Missing product-date combinations:",
      demand_calendar["demand"].isna().sum())

Complete calendar shape: (2939328, 3)

Original observed rows: 247902
Missing product-date combinations: 2691426


In [14]:
# Sort by product and date
product_history = daily_demand.sort_values(
    ["product_id", "date"]
).copy()

# Previous observed date for each product
product_history["previous_date"] = (
    product_history.groupby("product_id")["date"].shift(1)
)

# Days between consecutive observed dates
product_history["days_since_previous"] = (
    product_history["date"] -
    product_history["previous_date"]
).dt.days

# Check gap distribution
print("Gap between consecutive observed product dates:")

print(
    product_history["days_since_previous"]
    .dropna()
    .describe()
)

print("\nGaps of exactly 1 day:",
      (product_history["days_since_previous"] == 1).sum())

print("Gaps greater than 7 days:",
      (product_history["days_since_previous"] > 7).sum())

print("Gaps greater than 30 days:",
      (product_history["days_since_previous"] > 30).sum())

Gap between consecutive observed product dates:
count    243870.000000
mean          4.444372
std          11.510065
min           1.000000
25%           1.000000
50%           2.000000
75%           4.000000
max         407.000000
Name: days_since_previous, dtype: float64

Gaps of exactly 1 day: 109251
Gaps greater than 7 days: 27153
Gaps greater than 30 days: 4653


In [15]:
# Create previous-calendar-day demand
lag_1 = daily_demand[["date", "product_id", "demand"]].copy()

lag_1["date"] = lag_1["date"] + pd.Timedelta(days=1)

lag_1 = lag_1.rename(
    columns={"demand": "lag_1"}
)

# Join previous-day demand
daily_demand = daily_demand.merge(
    lag_1,
    on=["date", "product_id"],
    how="left"
)

print("Lag 1 created.")

print("\nNon-null Lag 1 values:",
      daily_demand["lag_1"].notna().sum())

print("Null Lag 1 values:",
      daily_demand["lag_1"].isna().sum())

print("\nSample:")
display(
    daily_demand[
        ["date", "product_id", "demand", "lag_1"]
    ].head(10)
)

C:\Users\rar95\AppData\Local\Temp\ipykernel_24312\515236348.py:4: DeprecationWarning: The 'generic' unit for NumPy timedelta is deprecated, and will raise an error in the future. This includes implicit conversion of bare integers (e.g. `+ 1`).Please use a specific unit instead.
  lag_1["date"] = lag_1["date"] + pd.Timedelta(days=1)


Lag 1 created.

Non-null Lag 1 values: 109251
Null Lag 1 values: 138651

Sample:


,date,product_id,demand,lag_1
0,2014-01-01,307,4,NaN
1,2014-01-01,521,1,NaN
2,2014-01-01,522,11,NaN
3,2014-01-01,547,1,NaN
4,2014-01-01,565,1,NaN
5,2014-01-01,601,4,NaN
6,2014-01-01,741,2,NaN
7,2014-01-01,803,4,NaN
8,2014-01-01,824,3,NaN
9,2014-01-01,837,3,NaN


In [16]:
for lag in [7, 14, 30]:
    
    lag_data = daily_demand[
        ["date", "product_id", "demand"]
    ].copy()
    
    lag_data["date"] = lag_data["date"] + pd.Timedelta(days=lag)
    
    lag_data = lag_data.rename(
        columns={"demand": f"lag_{lag}"}
    )
    
    daily_demand = daily_demand.merge(
        lag_data,
        on=["date", "product_id"],
        how="left"
    )

# Verify lag features
print("Lag features created:")
print(["lag_1", "lag_7", "lag_14", "lag_30"])

print("\nNon-null counts:")
print(
    daily_demand[
        ["lag_1", "lag_7", "lag_14", "lag_30"]
    ].notna().sum()
)

print("\nSample:")
display(
    daily_demand[
        ["date", "product_id", "demand",
         "lag_1", "lag_7", "lag_14", "lag_30"]
    ].head(10)
)

C:\Users\rar95\AppData\Local\Temp\ipykernel_24312\3245265139.py:7: DeprecationWarning: The 'generic' unit for NumPy timedelta is deprecated, and will raise an error in the future. This includes implicit conversion of bare integers (e.g. `+ 1`).Please use a specific unit instead.
  lag_data["date"] = lag_data["date"] + pd.Timedelta(days=lag)
C:\Users\rar95\AppData\Local\Temp\ipykernel_24312\3245265139.py:7: DeprecationWarning: The 'generic' unit for NumPy timedelta is deprecated, and will raise an error in the future. This includes implicit conversion of bare integers (e.g. `+ 1`).Please use a specific unit instead.
  lag_data["date"] = lag_data["date"] + pd.Timedelta(days=lag)
C:\Users\rar95\AppData\Local\Temp\ipykernel_24312\3245265139.py:7: DeprecationWarning: The 'generic' unit for NumPy timedelta is deprecated, and will raise an error in the future. This includes implicit conversion of bare integers (e.g. `+ 1`).Please use a specific unit instead.
  lag_data["date"] = lag_data["dat

Lag features created:
['lag_1', 'lag_7', 'lag_14', 'lag_30']

Non-null counts:
lag_1     109251
lag_7     120432
lag_14    112910
lag_30     86320
dtype: int64

Sample:


,date,product_id,demand,lag_1,lag_7,lag_14,lag_30
0,2014-01-01,307,4,NaN,NaN,NaN,NaN
1,2014-01-01,521,1,NaN,NaN,NaN,NaN
2,2014-01-01,522,11,NaN,NaN,NaN,NaN
3,2014-01-01,547,1,NaN,NaN,NaN,NaN
4,2014-01-01,565,1,NaN,NaN,NaN,NaN
5,2014-01-01,601,4,NaN,NaN,NaN,NaN
6,2014-01-01,741,2,NaN,NaN,NaN,NaN
7,2014-01-01,803,4,NaN,NaN,NaN,NaN
8,2014-01-01,824,3,NaN,NaN,NaN,NaN
9,2014-01-01,837,3,NaN,NaN,NaN,NaN


In [17]:
# Sort by product and date
daily_demand = daily_demand.sort_values(
    ["product_id", "date"]
).reset_index(drop=True)

# Previous observed demand
daily_demand["previous_demand"] = (
    daily_demand.groupby("product_id")["demand"]
    .shift(1)
)

# 7-observation rolling mean using only previous demand
daily_demand["rolling_mean_7"] = (
    daily_demand.groupby("product_id")["previous_demand"]
    .transform(
        lambda x: x.rolling(window=7, min_periods=1).mean()
    )
)

print("Rolling feature created.")

print("\nNon-null values:")
print(daily_demand["rolling_mean_7"].notna().sum())

print("\nSample:")
display(
    daily_demand[
        ["date", "product_id", "demand",
         "previous_demand", "rolling_mean_7"]
    ].head(15)
)

Rolling feature created.

Non-null values:
243870

Sample:


,date,product_id,demand,previous_demand,rolling_mean_7
0,2014-12-01,1,144,NaN,NaN
1,2014-12-02,1,50,144.0,144.000000
2,2014-12-03,1,26,50.0,97.000000
3,2014-12-05,1,12,26.0,73.333333
4,2014-12-06,1,48,12.0,58.000000
5,2014-12-07,1,36,48.0,56.000000
6,2014-12-08,1,50,36.0,52.666667
7,2014-12-09,1,24,50.0,52.285714
8,2014-12-12,1,12,24.0,35.142857
9,2014-12-13,1,12,12.0,29.714286


In [18]:
# 14-observation rolling mean
daily_demand["rolling_mean_14"] = (
    daily_demand.groupby("product_id")["previous_demand"]
    .transform(
        lambda x: x.rolling(window=14, min_periods=1).mean()
    )
)

# 30-observation rolling mean
daily_demand["rolling_mean_30"] = (
    daily_demand.groupby("product_id")["previous_demand"]
    .transform(
        lambda x: x.rolling(window=30, min_periods=1).mean()
    )
)

# 7-observation rolling standard deviation
daily_demand["rolling_std_7"] = (
    daily_demand.groupby("product_id")["previous_demand"]
    .transform(
        lambda x: x.rolling(window=7, min_periods=2).std()
    )
)

print("Rolling features created.")

print("\nNon-null counts:")
print(
    daily_demand[
        [
            "rolling_mean_7",
            "rolling_mean_14",
            "rolling_mean_30",
            "rolling_std_7"
        ]
    ].notna().sum()
)

print("\nSample:")
display(
    daily_demand[
        [
            "date",
            "product_id",
            "demand",
            "rolling_mean_7",
            "rolling_mean_14",
            "rolling_mean_30",
            "rolling_std_7"
        ]
    ].head(15)
)

Rolling features created.

Non-null counts:
rolling_mean_7     243870
rolling_mean_14    243870
rolling_mean_30    243870
rolling_std_7      240061
dtype: int64

Sample:


,date,product_id,demand,rolling_mean_7,rolling_mean_14,rolling_mean_30,rolling_std_7
0,2014-12-01,1,144,NaN,NaN,NaN,NaN
1,2014-12-02,1,50,144.000000,144.000000,144.000000,NaN
2,2014-12-03,1,26,97.000000,97.000000,97.000000,66.468037
3,2014-12-05,1,12,73.333333,73.333333,73.333333,62.364520
4,2014-12-06,1,48,58.000000,58.000000,58.000000,59.441848
5,2014-12-07,1,36,56.000000,56.000000,56.000000,51.672043
6,2014-12-08,1,50,52.666667,52.666667,52.666667,46.932576
7,2014-12-09,1,24,52.285714,52.285714,52.285714,42.855238
8,2014-12-12,1,12,35.142857,48.750000,48.750000,15.004761
9,2014-12-13,1,12,29.714286,44.666667,44.666667,15.596092


In [19]:
# Historical expanding mean
daily_demand["expanding_mean"] = (
    daily_demand.groupby("product_id")["previous_demand"]
    .transform(lambda x: x.expanding(min_periods=1).mean())
)

# Historical expanding standard deviation
daily_demand["expanding_std"] = (
    daily_demand.groupby("product_id")["previous_demand"]
    .transform(lambda x: x.expanding(min_periods=2).std())
)

# Number of previous observations
daily_demand["previous_count"] = (
    daily_demand.groupby("product_id")["previous_demand"]
    .transform(lambda x: x.notna().cumsum())
)

print("Expanding/history features created.")

print("\nNon-null counts:")
print(
    daily_demand[
        [
            "expanding_mean",
            "expanding_std",
            "previous_count"
        ]
    ].notna().sum()
)

print("\nSample:")
display(
    daily_demand[
        [
            "date",
            "product_id",
            "demand",
            "expanding_mean",
            "expanding_std",
            "previous_count"
        ]
    ].head(15)
)

Expanding/history features created.

Non-null counts:
expanding_mean    243870
expanding_std     240061
previous_count    247902
dtype: int64

Sample:


,date,product_id,demand,expanding_mean,expanding_std,previous_count
0,2014-12-01,1,144,NaN,NaN,0
1,2014-12-02,1,50,144.000000,NaN,1
2,2014-12-03,1,26,97.000000,66.468037,2
3,2014-12-05,1,12,73.333333,62.364520,3
4,2014-12-06,1,48,58.000000,59.441848,4
5,2014-12-07,1,36,56.000000,51.672043,5
6,2014-12-08,1,50,52.666667,46.932576,6
7,2014-12-09,1,24,52.285714,42.855238,7
8,2014-12-12,1,12,48.750000,40.917164,8
9,2014-12-13,1,12,44.666667,40.187063,9


In [21]:
# Historical cumulative demand for each product
daily_demand["product_total_demand"] = (
    daily_demand.groupby("product_id")["previous_demand"]
    .transform(lambda x: x.expanding(min_periods=1).sum())
)

# Historical average demand for each product
daily_demand["product_avg_demand"] = (
    daily_demand.groupby("product_id")["previous_demand"]
    .transform(lambda x: x.expanding(min_periods=1).mean())
)

print("Product-level historical features corrected.")

print("\nNon-null counts:")
print(
    daily_demand[
        [
            "product_total_demand",
            "product_avg_demand"
        ]
    ].notna().sum()
)

print("\nSample:")
display(
    daily_demand[
        [
            "date",
            "product_id",
            "demand",
            "previous_demand",
            "product_total_demand",
            "product_avg_demand"
        ]
    ].head(15)
)

Product-level historical features corrected.

Non-null counts:
product_total_demand    243870
product_avg_demand      243870
dtype: int64

Sample:


,date,product_id,demand,previous_demand,product_total_demand,product_avg_demand
0,2014-12-01,1,144,NaN,NaN,NaN
1,2014-12-02,1,50,144.0,144.0,144.000000
2,2014-12-03,1,26,50.0,194.0,97.000000
3,2014-12-05,1,12,26.0,220.0,73.333333
4,2014-12-06,1,48,12.0,232.0,58.000000
5,2014-12-07,1,36,48.0,280.0,56.000000
6,2014-12-08,1,50,36.0,316.0,52.666667
7,2014-12-09,1,24,50.0,366.0,52.285714
8,2014-12-12,1,12,24.0,390.0,48.750000
9,2014-12-13,1,12,12.0,402.0,44.666667


In [22]:
# Customer-product relationship
customer_product_stats = (
    purchases_clean
    .groupby("product_id")["CustomerID"]
    .nunique()
)

print("Products:", customer_product_stats.shape[0])

print("\nUnique customers per product:")
print(customer_product_stats.describe())

print("\nProducts with only 1 customer:",
      (customer_product_stats == 1).sum())

print("Products with <= 5 customers:",
      (customer_product_stats <= 5).sum())

print("Products with > 100 customers:",
      (customer_product_stats > 100).sum())

Products: 4032

Unique customers per product:
count    4032.000000
mean       75.132688
std       109.582444
min         1.000000
25%         9.000000
50%        38.000000
75%        98.000000
max      1786.000000
Name: CustomerID, dtype: float64

Products with only 1 customer: 226
Products with <= 5 customers: 717
Products with > 100 customers: 984


In [23]:
# Create daily unique customers per product
daily_customers = (
    purchases_clean
    .groupby(["date", "product_id"])["CustomerID"]
    .nunique()
    .reset_index(name="daily_unique_customers")
)

# Historical cumulative customer count
daily_customers = daily_customers.sort_values(
    ["product_id", "date"]
)

daily_customers["product_unique_customers"] = (
    daily_customers
    .groupby("product_id")["daily_unique_customers"]
    .cumsum()
)

# Shift so current day's customers are not included
daily_customers["product_unique_customers"] = (
    daily_customers
    .groupby("product_id")["product_unique_customers"]
    .shift(1)
)

# Merge into modeling dataset
daily_demand = daily_demand.merge(
    daily_customers[
        ["date", "product_id", "product_unique_customers"]
    ],
    on=["date", "product_id"],
    how="left"
)

print("Feature created.")

print("\nNon-null values:")
print(
    daily_demand["product_unique_customers"].notna().sum()
)

print("\nSample:")
display(
    daily_demand[
        [
            "date",
            "product_id",
            "demand",
            "product_unique_customers"
        ]
    ].head(15)
)

Feature created.

Non-null values:
243870

Sample:


,date,product_id,demand,product_unique_customers
0,2014-12-01,1,144,NaN
1,2014-12-02,1,50,3.0
2,2014-12-03,1,26,7.0
3,2014-12-05,1,12,9.0
4,2014-12-06,1,48,10.0
5,2014-12-07,1,36,11.0
6,2014-12-08,1,50,13.0
7,2014-12-09,1,24,16.0
8,2014-12-12,1,12,17.0
9,2014-12-13,1,12,18.0


In [24]:
# Sort customer transactions
customer_history = (
    purchases_clean[
        ["date", "product_id", "CustomerID"]
    ]
    .drop_duplicates()
    .sort_values(["product_id", "date"])
)

# Keep only the first purchase date of each customer-product pair
first_customer_purchase = (
    customer_history
    .groupby(["product_id", "CustomerID"])["date"]
    .min()
    .reset_index()
)

# Number of new customers acquired on each date
new_customers = (
    first_customer_purchase
    .groupby(["date", "product_id"])
    .size()
    .reset_index(name="new_customers")
)

# Sort by product and date
new_customers = new_customers.sort_values(
    ["product_id", "date"]
)

# Cumulative distinct customers
new_customers["product_unique_customers"] = (
    new_customers
    .groupby("product_id")["new_customers"]
    .cumsum()
)

# Shift so current-day customers are excluded
new_customers["product_unique_customers"] = (
    new_customers
    .groupby("product_id")["product_unique_customers"]
    .shift(1)
)

# Remove the previous incorrect feature
daily_demand = daily_demand.drop(
    columns=["product_unique_customers"]
)

# Merge corrected feature
daily_demand = daily_demand.merge(
    new_customers[
        ["date", "product_id", "product_unique_customers"]
    ],
    on=["date", "product_id"],
    how="left"
)

print("Corrected historical unique-customer feature.")

print("\nNon-null values:")
print(
    daily_demand["product_unique_customers"].notna().sum()
)

print("\nSample:")
display(
    daily_demand[
        [
            "date",
            "product_id",
            "demand",
            "product_unique_customers"
        ]
    ].head(15)
)

Corrected historical unique-customer feature.

Non-null values:
200434

Sample:


,date,product_id,demand,product_unique_customers
0,2014-12-01,1,144,NaN
1,2014-12-02,1,50,3.0
2,2014-12-03,1,26,7.0
3,2014-12-05,1,12,9.0
4,2014-12-06,1,48,10.0
5,2014-12-07,1,36,11.0
6,2014-12-08,1,50,13.0
7,2014-12-09,1,24,16.0
8,2014-12-12,1,12,17.0
9,2014-12-13,1,12,18.0


In [25]:
from pathlib import Path

raw_path = PROJECT_ROOT / "data" / "raw"

for file in raw_path.glob("*.csv"):
    df = pd.read_csv(file, nrows=5)
    
    print(f"\n{file.name}")
    print("Columns:", df.columns.tolist())


customers.csv
Columns: ['CustomerID', 'customer_type']

invoice_items.csv
Columns: ['InvoiceID', 'product_id', 'quantity', 'price', 'line_total']

products.csv
Columns: ['product_id', 'item', 'category', 'price']

purchases.csv
Columns: ['InvoiceID', 'date', 'CustomerID', 'product_id', 'quantity']


In [26]:
# Load only the required columns
invoice_items = pd.read_csv(
    raw_path / "invoice_items.csv",
    usecols=["InvoiceID", "product_id", "quantity", "price", "line_total"]
)

products = pd.read_csv(
    raw_path / "products.csv",
    usecols=["product_id", "item", "category", "price"]
)

# Check price information
print("Invoice items:")
print(invoice_items[["price", "line_total"]].describe())

print("\nMissing values:")
print(invoice_items[["price", "line_total"]].isnull().sum())

print("\nProducts:")
print(products["price"].describe())

print("\nMissing product prices:")
print(products["price"].isnull().sum())

# Check whether products have multiple prices in invoice_items
price_counts = (
    invoice_items
    .groupby("product_id")["price"]
    .nunique()
)

print("\nProducts with multiple transaction prices:",
      (price_counts > 1).sum())

print("Products with exactly one transaction price:",
      (price_counts == 1).sum())

Invoice items:
              price     line_total
count  436689.00000  436689.000000
mean        3.22677      21.374055
std        21.11145     295.049213
min         0.00000       0.000000
25%         1.25000       4.680000
50%         2.08000      10.760000
75%         3.75000      18.990000
max      8142.75000  168469.600000

Missing values:
price         0
line_total    0
dtype: int64

Products:
count    4033.000000
mean        3.798143
std        13.059295
min         0.001000
25%         1.250000
50%         2.100000
75%         4.150000
max       649.500000
Name: price, dtype: float64

Missing product prices:
0

Products with multiple transaction prices: 2714
Products with exactly one transaction price: 1319


In [27]:
# Check InvoiceID + product_id uniqueness in invoice_items
invoice_item_counts = (
    invoice_items
    .groupby(["InvoiceID", "product_id"])
    .size()
)

print("InvoiceID + product_id combinations:",
      len(invoice_item_counts))

print("Repeated InvoiceID + product_id combinations:",
      (invoice_item_counts > 1).sum())

# Compare with cleaned transaction combinations
clean_combinations = (
    purchases_clean[["InvoiceID", "product_id"]]
    .drop_duplicates()
)

print("\nUnique cleaned InvoiceID + product_id combinations:",
      len(clean_combinations))

# Check how many cleaned combinations exist in invoice_items
invoice_keys = (
    invoice_items[["InvoiceID", "product_id"]]
    .drop_duplicates()
)

matched = clean_combinations.merge(
    invoice_keys,
    on=["InvoiceID", "product_id"],
    how="left",
    indicator=True
)

print("\nCleaned combinations matched in invoice_items:")
print((matched["_merge"] == "both").sum())

print("Cleaned combinations NOT matched:")
print((matched["_merge"] == "left_only").sum())

InvoiceID + product_id combinations: 425701
Repeated InvoiceID + product_id combinations: 10007

Unique cleaned InvoiceID + product_id combinations: 425699

Cleaned combinations matched in invoice_items:
425699
Cleaned combinations NOT matched:
0


In [28]:
# Check whether repeated InvoiceID + product_id combinations
# have consistent prices

price_consistency = (
    invoice_items
    .groupby(["InvoiceID", "product_id"])["price"]
    .nunique()
)

print("Combinations with exactly one price:",
      (price_consistency == 1).sum())

print("Combinations with multiple prices:",
      (price_consistency > 1).sum())

print("\nMaximum number of prices for one InvoiceID + product_id:",
      price_consistency.max())

Combinations with exactly one price: 425483
Combinations with multiple prices: 218

Maximum number of prices for one InvoiceID + product_id: 4


In [29]:
# Find InvoiceID + product_id combinations with multiple prices
multi_price_keys = (
    price_consistency[price_consistency > 1]
    .reset_index()[["InvoiceID", "product_id"]]
)

# Show the affected records
multi_price_examples = (
    invoice_items
    .merge(
        multi_price_keys,
        on=["InvoiceID", "product_id"],
        how="inner"
    )
    .sort_values(["InvoiceID", "product_id", "price"])
)

print("Affected combinations:", len(multi_price_keys))

print("\nAffected rows:", len(multi_price_examples))

display(
    multi_price_examples.head(30)
)

Affected combinations: 218

Affected rows: 457


,InvoiceID,product_id,quantity,price,line_total
0,536569,2061,1,1.25,1.25
1,536569,2061,1,18.95,18.95
2,536987,852,1,0.65,0.65
3,536987,852,1,0.85,0.85
4,536987,852,1,1.25,1.25
6,537126,849,1,0.85,0.85
5,537126,849,1,1.25,1.25
7,537140,2061,1,0.42,0.42
8,537140,2061,1,0.85,0.85
9,537236,2848,16,3.39,54.24


In [30]:
# Quantity-weighted price for each InvoiceID + product_id
invoice_price = (
    invoice_items
    .assign(
        value=lambda x: x["quantity"] * x["price"]
    )
    .groupby(["InvoiceID", "product_id"], as_index=False)
    .agg(
        total_quantity=("quantity", "sum"),
        total_value=("value", "sum")
    )
)

invoice_price["transaction_price"] = (
    invoice_price["total_value"] /
    invoice_price["total_quantity"]
)

# Verify
print("Price lookup rows:", len(invoice_price))

print("\nMissing transaction prices:",
      invoice_price["transaction_price"].isna().sum())

print("\nTransaction price summary:")
print(invoice_price["transaction_price"].describe())

print("\nSample:")
display(invoice_price.head(10))

Price lookup rows: 425701

Missing transaction prices: 0

Transaction price summary:
count    425701.000000
mean          3.232650
std          20.379595
min           0.000000
25%           1.250000
50%           2.080000
75%           3.750000
max        8142.750000
Name: transaction_price, dtype: float64

Sample:


,InvoiceID,product_id,total_quantity,total_value,transaction_price
0,1,3029,1,11.62,11.62
1,1,3046,3,6.51,2.17
2,1,3891,3,6.57,2.19
3,1,4004,1,4.49,4.49
4,2,2464,2,7.54,3.77
5,2,3025,2,2.00,1.00
6,2,3891,4,8.76,2.19
7,3,613,3,23.91,7.97
8,3,2163,1,2.19,2.19
9,4,1734,3,19.08,6.36


In [31]:
# Add transaction price to cleaned transactions
purchases_with_price = purchases_clean.merge(
    invoice_price[
        ["InvoiceID", "product_id", "transaction_price"]
    ],
    on=["InvoiceID", "product_id"],
    how="left"
)

# Verify the merge
print("Rows:", len(purchases_with_price))
print(
    "Missing transaction prices:",
    purchases_with_price["transaction_price"].isna().sum()
)

# Calculate transaction value
purchases_with_price["transaction_value"] = (
    purchases_with_price["quantity"] *
    purchases_with_price["transaction_price"]
)

# Calculate daily product price
daily_price = (
    purchases_with_price
    .groupby(["date", "product_id"], as_index=False)
    .agg(
        total_quantity=("quantity", "sum"),
        total_value=("transaction_value", "sum")
    )
)

daily_price["daily_avg_price"] = (
    daily_price["total_value"] /
    daily_price["total_quantity"]
)

print("\nDaily product-price rows:", len(daily_price))

print("\nMissing daily prices:",
      daily_price["daily_avg_price"].isna().sum())

print("\nDaily average price summary:")
print(daily_price["daily_avg_price"].describe())

print("\nSample:")
display(
    daily_price[
        ["date", "product_id", "daily_avg_price"]
    ].head(10)
)

Rows: 431171
Missing transaction prices: 0

Daily product-price rows: 247902

Missing daily prices: 0

Daily average price summary:
count    247902.000000
mean          3.237920
std          15.277304
min           0.000000
25%           1.250000
50%           1.950000
75%           3.750000
max        4161.060000
Name: daily_avg_price, dtype: float64

Sample:


,date,product_id,daily_avg_price
0,2014-01-01,307,4.28
1,2014-01-01,521,2.47
2,2014-01-01,522,7.61
3,2014-01-01,547,2.19
4,2014-01-01,565,4.84
5,2014-01-01,601,3.57
6,2014-01-01,741,8.11
7,2014-01-01,803,3.56
8,2014-01-01,824,3.94
9,2014-01-01,837,3.37


In [32]:
# Sort daily prices
daily_price = daily_price.sort_values(
    ["product_id", "date"]
).reset_index(drop=True)

# Previous observed price
daily_price["previous_price"] = (
    daily_price
    .groupby("product_id")["daily_avg_price"]
    .shift(1)
)

# Price change from previous observed price
daily_price["price_change"] = (
    daily_price["daily_avg_price"] -
    daily_price["previous_price"]
)

# Merge historical price features
daily_demand = daily_demand.merge(
    daily_price[
        [
            "date",
            "product_id",
            "previous_price",
            "price_change"
        ]
    ],
    on=["date", "product_id"],
    how="left"
)

print("Historical price features created.")

print("\nNon-null counts:")
print(
    daily_demand[
        ["previous_price", "price_change"]
    ].notna().sum()
)

print("\nSample:")
display(
    daily_demand[
        [
            "date",
            "product_id",
            "demand",
            "previous_price",
            "price_change"
        ]
    ].head(15)
)

Historical price features created.

Non-null counts:
previous_price    243870
price_change      243870
dtype: int64

Sample:


,date,product_id,demand,previous_price,price_change
0,2014-12-01,1,144,NaN,NaN
1,2014-12-02,1,50,0.85,0.000000e+00
2,2014-12-03,1,26,0.85,-1.110223e-16
3,2014-12-05,1,12,0.85,1.110223e-16
4,2014-12-06,1,48,0.85,0.000000e+00
5,2014-12-07,1,36,0.85,0.000000e+00
6,2014-12-08,1,50,0.85,0.000000e+00
7,2014-12-09,1,24,0.85,0.000000e+00
8,2014-12-12,1,12,0.85,0.000000e+00
9,2014-12-13,1,12,0.85,0.000000e+00


In [33]:
# Create historical price change without using current-day price
daily_price["price_change"] = (
    daily_price
    .groupby("product_id")["daily_avg_price"]
    .shift(1)
    -
    daily_price
    .groupby("product_id")["daily_avg_price"]
    .shift(2)
)

# Replace the existing price_change in daily_demand
daily_demand = daily_demand.drop(
    columns=["price_change"]
)

daily_demand = daily_demand.merge(
    daily_price[
        ["date", "product_id", "price_change"]
    ],
    on=["date", "product_id"],
    how="left"
)

print("Historical price change corrected.")

print("\nNon-null counts:")
print(
    daily_demand[
        ["previous_price", "price_change"]
    ].notna().sum()
)

print("\nSample:")
display(
    daily_demand[
        [
            "date",
            "product_id",
            "demand",
            "previous_price",
            "price_change"
        ]
    ].head(15)
)

Historical price change corrected.

Non-null counts:
previous_price    243870
price_change      240061
dtype: int64

Sample:


,date,product_id,demand,previous_price,price_change
0,2014-12-01,1,144,NaN,NaN
1,2014-12-02,1,50,0.85,NaN
2,2014-12-03,1,26,0.85,0.000000e+00
3,2014-12-05,1,12,0.85,-1.110223e-16
4,2014-12-06,1,48,0.85,1.110223e-16
5,2014-12-07,1,36,0.85,0.000000e+00
6,2014-12-08,1,50,0.85,0.000000e+00
7,2014-12-09,1,24,0.85,0.000000e+00
8,2014-12-12,1,12,0.85,0.000000e+00
9,2014-12-13,1,12,0.85,0.000000e+00


In [34]:
# Current feature set
feature_columns = [
    "year",
    "month",
    "day",
    "day_of_week",
    "week_of_year",
    "quarter",
    "day_of_year",
    "is_weekend",
    "lag_1",
    "lag_7",
    "lag_14",
    "lag_30",
    "rolling_mean_7",
    "rolling_mean_14",
    "rolling_mean_30",
    "rolling_std_7",
    "expanding_mean",
    "expanding_std",
    "previous_count",
    "product_total_demand",
    "product_avg_demand",
    "product_unique_customers",
    "previous_price",
    "price_change"
]

print("Target:", "demand")

print("\nNumber of features:", len(feature_columns))

print("\nFeature columns:")
for feature in feature_columns:
    print("-", feature)

print("\nCurrent dataset shape:", daily_demand.shape)

print("\nCurrent demand summary:")
print(daily_demand["demand"].describe())

Target: demand

Number of features: 24

Feature columns:
- year
- month
- day
- day_of_week
- week_of_year
- quarter
- day_of_year
- is_weekend
- lag_1
- lag_7
- lag_14
- lag_30
- rolling_mean_7
- rolling_mean_14
- rolling_mean_30
- rolling_std_7
- expanding_mean
- expanding_std
- previous_count
- product_total_demand
- product_avg_demand
- product_unique_customers
- previous_price
- price_change

Current dataset shape: (247902, 28)

Current demand summary:
count    247902.000000
mean         20.597248
std          65.845481
min           1.000000
25%           3.000000
50%           7.000000
75%          20.000000
max       12540.000000
Name: demand, dtype: float64


In [35]:
# Check missing values in target and features
nan_summary = (
    daily_demand[["demand"] + feature_columns]
    .isna()
    .sum()
    .sort_values(ascending=False)
)

print("Missing values by column:")
print(nan_summary[nan_summary > 0])

print("\nTotal rows:", len(daily_demand))

print("\nRows with any missing feature:")
print(
    daily_demand[feature_columns]
    .isna()
    .any(axis=1)
    .sum()
)

Missing values by column:
lag_30                      161582
lag_1                       138651
lag_14                      134992
lag_7                       127470
product_unique_customers     47468
price_change                  7841
expanding_std                 7841
rolling_std_7                 7841
rolling_mean_30               4032
product_avg_demand            4032
rolling_mean_14               4032
rolling_mean_7                4032
expanding_mean                4032
previous_price                4032
product_total_demand          4032
dtype: int64

Total rows: 247902

Rows with any missing feature:
225771


In [36]:
# Start from first purchase date of each customer-product pair
first_customer_purchase = (
    purchases_clean[
        ["date", "product_id", "CustomerID"]
    ]
    .drop_duplicates()
    .groupby(["product_id", "CustomerID"])["date"]
    .min()
    .reset_index()
)

# Count new customers acquired by each product on each date
new_customers = (
    first_customer_purchase
    .groupby(["date", "product_id"])
    .size()
    .reset_index(name="new_customers")
)

# Create cumulative distinct customer count
new_customers = new_customers.sort_values(
    ["product_id", "date"]
)

new_customers["product_unique_customers"] = (
    new_customers
    .groupby("product_id")["new_customers"]
    .cumsum()
)

# Shift so current-day new customers are excluded
new_customers["product_unique_customers"] = (
    new_customers
    .groupby("product_id")["product_unique_customers"]
    .shift(1)
)

# Merge onto every observed product-date
daily_demand = daily_demand.drop(
    columns=["product_unique_customers"]
)

daily_demand = daily_demand.merge(
    new_customers[
        ["date", "product_id", "product_unique_customers"]
    ],
    on=["date", "product_id"],
    how="left"
)

# Carry historical customer count forward within each product
daily_demand["product_unique_customers"] = (
    daily_demand
    .sort_values(["product_id", "date"])
    .groupby("product_id")["product_unique_customers"]
    .ffill()
)

print("Corrected historical unique-customer feature.")

print("\nMissing values:")
print(
    daily_demand["product_unique_customers"].isna().sum()
)

print("\nNon-null values:")
print(
    daily_demand["product_unique_customers"].notna().sum()
)

print("\nSample:")
display(
    daily_demand[
        [
            "date",
            "product_id",
            "demand",
            "product_unique_customers"
        ]
    ].head(15)
)

Corrected historical unique-customer feature.

Missing values:
4156

Non-null values:
243746

Sample:


,date,product_id,demand,product_unique_customers
0,2014-12-01,1,144,NaN
1,2014-12-02,1,50,3.0
2,2014-12-03,1,26,7.0
3,2014-12-05,1,12,9.0
4,2014-12-06,1,48,10.0
5,2014-12-07,1,36,11.0
6,2014-12-08,1,50,13.0
7,2014-12-09,1,24,16.0
8,2014-12-12,1,12,17.0
9,2014-12-13,1,12,18.0


In [37]:
nan_summary = (
    daily_demand[["demand"] + feature_columns]
    .isna()
    .sum()
    .sort_values(ascending=False)
)

print("Missing values by column:")
print(nan_summary[nan_summary > 0])

print("\nRows with any missing feature:")
print(
    daily_demand[feature_columns]
    .isna()
    .any(axis=1)
    .sum()
)

Missing values by column:
lag_30                      161582
lag_1                       138651
lag_14                      134992
lag_7                       127470
rolling_std_7                 7841
expanding_std                 7841
price_change                  7841
product_unique_customers      4156
rolling_mean_30               4032
product_avg_demand            4032
rolling_mean_14               4032
rolling_mean_7                4032
expanding_mean                4032
previous_price                4032
product_total_demand          4032
dtype: int64

Rows with any missing feature:
220815


In [38]:
fill_zero_cols = [
    "lag_1",
    "lag_7",
    "lag_14",
    "lag_30",
    "rolling_mean_7",
    "rolling_mean_14",
    "rolling_mean_30",
    "rolling_std_7",
    "expanding_mean",
    "expanding_std",
    "previous_count",
    "product_total_demand",
    "product_avg_demand",
    "product_unique_customers",
    "previous_price",
    "price_change"
]

daily_demand[fill_zero_cols] = (
    daily_demand[fill_zero_cols]
    .fillna(0)
)

remaining_nans = (
    daily_demand[feature_columns]
    .isna()
    .sum()
    .sum()
)

print("Remaining feature NaNs:", remaining_nans)

print("\nDataset shape:")
print(daily_demand.shape)

Remaining feature NaNs: 0

Dataset shape:
(247902, 28)


#### The interpretation: a lag filled with 0 now represents "no observed prior demand", not necessarily proven zero sales.

In [39]:
# Final feature validation

print("Dataset shape:", daily_demand.shape)

# 1. Check required columns
required_columns = [
    "date",
    "product_id",
    "demand"
] + feature_columns

missing_columns = [
    col for col in required_columns
    if col not in daily_demand.columns
]

print("\nMissing required columns:", missing_columns)

# 2. Check missing values
print(
    "\nTotal missing values:",
    daily_demand[required_columns].isna().sum().sum()
)

# 3. Check duplicate date-product rows
duplicates = daily_demand.duplicated(
    subset=["date", "product_id"]
).sum()

print("Duplicate date-product rows:", duplicates)

# 4. Check target
print("\nInvalid target values:")
print("Zero demand:", (daily_demand["demand"] == 0).sum())
print("Negative demand:", (daily_demand["demand"] < 0).sum())

# 5. Check infinite values
numeric_features = daily_demand[feature_columns].select_dtypes(
    include="number"
)

print("\nInfinite feature values:",
      numeric_features.isin([float("inf"), float("-inf")]).sum().sum())

# 6. Check date ordering
print("\nDate range:")
print(daily_demand["date"].min(), "to", daily_demand["date"].max())

# 7. Final feature count
print("\nNumber of features:", len(feature_columns))

print("\nFeature-engineered dataset preview:")
display(
    daily_demand[
        ["date", "product_id", "demand"] + feature_columns
    ].head()
)

Dataset shape: (247902, 28)

Missing required columns: []

Total missing values: 0
Duplicate date-product rows: 0

Invalid target values:
Zero demand: 0
Negative demand: 0

Infinite feature values: 0

Date range:
2014-01-01 00:00:00 to 2015-12-30 00:00:00

Number of features: 24

Feature-engineered dataset preview:


,date,product_id,demand,year,month,day,day_of_week,week_of_year,quarter,day_of_year,...,rolling_mean_30,rolling_std_7,expanding_mean,expanding_std,previous_count,product_total_demand,product_avg_demand,product_unique_customers,previous_price,price_change
0,2014-12-01,1,144,2014,12,1,0,49,4,335,...,0.000000,0.000000,0.000000,0.000000,0,0.0,0.000000,0.0,0.00,0.000000e+00
1,2014-12-02,1,50,2014,12,2,1,49,4,336,...,144.000000,0.000000,144.000000,0.000000,1,144.0,144.000000,3.0,0.85,0.000000e+00
2,2014-12-03,1,26,2014,12,3,2,49,4,337,...,97.000000,66.468037,97.000000,66.468037,2,194.0,97.000000,7.0,0.85,0.000000e+00
3,2014-12-05,1,12,2014,12,5,4,49,4,339,...,73.333333,62.364520,73.333333,62.364520,3,220.0,73.333333,9.0,0.85,-1.110223e-16
4,2014-12-06,1,48,2014,12,6,5,49,4,340,...,58.000000,59.441848,58.000000,59.441848,4,232.0,58.000000,10.0,0.85,1.110223e-16


In [40]:
# Remove helper column
daily_demand = daily_demand.drop(
    columns=["previous_demand"]
)

# Remove floating-point noise from price change
daily_demand["price_change"] = (
    daily_demand["price_change"].round(6)
)

# Final validation
print("Final dataset shape:", daily_demand.shape)

print("\nExpected columns:", 3 + len(feature_columns))
print("Actual columns:", len(daily_demand.columns))

print("\nRemaining NaNs:",
      daily_demand.isna().sum().sum())

print("Duplicate date-product rows:",
      daily_demand.duplicated(
          subset=["date", "product_id"]
      ).sum())

print("\nPrice change sample:")
print(
    daily_demand["price_change"]
    .head(15)
    .tolist()
)

print("\nFinal columns:")
print(daily_demand.columns.tolist())

Final dataset shape: (247902, 27)

Expected columns: 27
Actual columns: 27

Remaining NaNs: 0
Duplicate date-product rows: 0

Price change sample:
[0.0, 0.0, 0.0, -0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0]

Final columns:
['date', 'product_id', 'demand', 'year', 'month', 'day', 'day_of_week', 'week_of_year', 'quarter', 'day_of_year', 'is_weekend', 'lag_1', 'lag_7', 'lag_14', 'lag_30', 'rolling_mean_7', 'rolling_mean_14', 'rolling_mean_30', 'rolling_std_7', 'expanding_mean', 'expanding_std', 'previous_count', 'product_total_demand', 'product_avg_demand', 'previous_price', 'price_change', 'product_unique_customers']


##### Save feature-engineered dataset

In [41]:
# Save feature-engineered dataset
features_path = PROJECT_ROOT / "data" / "processed" / "purchases_features.csv"

daily_demand.to_csv(
    features_path,
    index=False
)

print("Feature-engineered dataset saved successfully.")
print("Path:", features_path)
print("Shape:", daily_demand.shape)

Feature-engineered dataset saved successfully.
Path: d:\All ML Projects\Retail_Demand_Forecasting\data\processed\purchases_features.csv
Shape: (247902, 27)


In [42]:
print("File exists:", features_path.exists())
print("File size (MB):", round(features_path.stat().st_size / (1024 * 1024), 2))

File exists: True
File size (MB): 46.2
